In [1]:
import win32com.client
import win32clipboard
import win32gui
import win32con
import pandas as pd
import subprocess
import time
from pathlib import Path

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

RAW_DIR  = BASE_PATH / "data" / "raw"
DIM_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

NOME_REAL        = "EXPORT_REAL.XLSX"
NOME_COMPROMISSO = "EXPORT_COMPROMIS.XLSX"

RAW_DIR.mkdir(parents=True, exist_ok=True)

# ======================================================
# HELPERS
# ======================================================

def set_clipboard_sap(lista_ordens):
    texto = "\r\n".join(lista_ordens)
    win32clipboard.OpenClipboard()
    try:
        win32clipboard.EmptyClipboard()
        win32clipboard.SetClipboardData(win32clipboard.CF_UNICODETEXT, texto)
    finally:
        win32clipboard.CloseClipboard()


def abrir_nova_sessao_sap(session_atual):
    """
    Abre nova sessão SAP (Ctrl+N) via sendVKey(74)
    Compatível com qualquer SAP GUI corporativo.
    """
    connection = session_atual.Parent
    qtd_antes = connection.Children.Count

    # Ctrl + N
    session_atual.findById("wnd[0]").sendVKey(74)
    time.sleep(2)

    while connection.Children.Count <= qtd_antes:
        time.sleep(0.5)

    nova_sessao = connection.Children(connection.Children.Count - 1)
    nova_sessao.findById("wnd[0]").maximize()
    nova_sessao.findById("wnd[0]").setFocus()

    print("   🆕 Nova sessão SAP aberta para COMPROMISSO.")
    return nova_sessao


def fechar_excel_e_focar_sap(session):
    subprocess.run(
        ["taskkill", "/f", "/im", "EXCEL.EXE"],
        capture_output=True,
        shell=True
    )
    time.sleep(2)

    def _focar(hwnd, _):
        if win32gui.IsWindowVisible(hwnd):
            titulo = win32gui.GetWindowText(hwnd)
            if "SAP" in titulo:
                win32gui.ShowWindow(hwnd, win32con.SW_MAXIMIZE)
                win32gui.SetForegroundWindow(hwnd)

    try:
        win32gui.EnumWindows(_focar, None)
    except:
        pass

    time.sleep(1)

    try:
        session.findById("wnd[0]").maximize()
        session.findById("wnd[0]").setFocus()
    except:
        pass

    time.sleep(1)
    print("   📊 Excel fechado. SAP maximizado e em foco.")


def exportar_sap_direto(session, nome_arquivo):
    try:
        grid_id = "wnd[0]/usr/cntlGRID1/shellcont/shell/shellcont[1]/shell"
        shell = session.findById(grid_id)
        shell.contextMenu()
        shell.selectContextMenuItem("&XXL")
        time.sleep(2)

        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(1)

        session.findById("wnd[1]/usr/ctxtDY_PATH").text      = str(RAW_DIR)
        session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = nome_arquivo
        session.findById("wnd[1]").sendVKey(0)

        time.sleep(4)
        print(f"   ✅ {nome_arquivo} exportado.")

    except Exception as e:
        print(f"   ⚠️ Erro na exportação ({nome_arquivo}): {e}")


def resumo_arquivo(caminho: Path, label: str):
    try:
        df = pd.read_excel(caminho)
        col_ordem = next((c for c in df.columns if "ordem" in str(c).lower()), None)
        col_valor = next((c for c in df.columns if "valor" in str(c).lower()), None)

        qtde  = df[col_ordem].nunique() if col_ordem else len(df)
        total = pd.to_numeric(df[col_valor], errors="coerce").sum() if col_valor else 0

        fmt = f"R$ {total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

        print(f"\n   📋 {label}")
        print(f"      Ordens únicas : {qtde:>6}")
        print(f"      Valor total   : {fmt}")

    except Exception as e:
        print(f"   ⚠️ Erro ao ler {label}: {e}")

# ======================================================
# MAIN
# ======================================================

def run_rpa():
    print("=" * 55)
    print("🚀  EXTRAÇÃO EM MASSA SAP — ZCO004")
    print("=" * 55)

    df_site = pd.read_excel(DIM_SITE)
    ordens  = df_site["ordem"].dropna().astype(str).str.strip().tolist()
    print(f"\n📂 dim_site carregado: {len(ordens)} ordens")

    sap = win32com.client.GetObject("SAPGUI")
    session_real = sap.GetScriptingEngine.Children(0).Children(0)
    session_real.findById("wnd[0]").maximize()

    # ================= REAL =================
    session_real.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
    session_real.findById("wnd[0]").sendVKey(0)
    session_real.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

    session_real.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
    set_clipboard_sap(ordens)
    session_real.findById("wnd[1]/tbar[0]/btn[24]").press()
    session_real.findById("wnd[1]/tbar[0]/btn[8]").press()
    session_real.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(6)
    print("\n📦 Extraindo REAL...")

    session_real.findById("wnd[0]/usr/lbl[62,6]").setFocus()
    session_real.findById("wnd[0]/tbar[1]/btn[7]").press()
    session_real.findById("wnd[1]/usr/lbl[1,3]").setFocus()
    session_real.findById("wnd[1]").sendVKey(2)

    exportar_sap_direto(session_real, NOME_REAL)
    fechar_excel_e_focar_sap(session_real)
    session_real.findById("wnd[0]").sendVKey(12)

    # ================= COMPROMISSO =================
    print("\n📦 Extraindo COMPROMISSO em nova sessão...")
    session_comp = abrir_nova_sessao_sap(session_real)

    try:
        session_comp.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
        session_comp.findById("wnd[0]").sendVKey(0)
        session_comp.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

        session_comp.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
        set_clipboard_sap(ordens)
        session_comp.findById("wnd[1]/tbar[0]/btn[24]").press()
        session_comp.findById("wnd[1]/tbar[0]/btn[8]").press()
        session_comp.findById("wnd[0]/tbar[1]/btn[8]").press()

        time.sleep(10)

        session_comp.findById("wnd[0]/usr/lbl[78,6]").setFocus()
        session_comp.findById("wnd[0]/tbar[1]/btn[7]").press()
        session_comp.findById("wnd[1]/usr/lbl[1,1]").setFocus()
        session_comp.findById("wnd[1]").sendVKey(2)

        exportar_sap_direto(session_comp, NOME_COMPROMISSO)
        fechar_excel_e_focar_sap(session_comp)

    finally:
        session_comp.findById("wnd[0]").close()
        print("   🧹 Sessão de COMPROMISSO encerrada.")

    # ================= RESUMO =================
    print("\n" + "=" * 55)
    print("📊  RESUMO DA EXTRAÇÃO")
    print("=" * 55)

    resumo_arquivo(RAW_DIR / NOME_REAL,        "REAL        (EXPORT_REAL.XLSX)")
    resumo_arquivo(RAW_DIR / NOME_COMPROMISSO, "COMPROMISSO (EXPORT_COMPROMIS.XLSX)")

    print("\n" + "=" * 55)
    print("✅  Processo finalizado!")
    print("=" * 55)


if __name__ == "__main__":
    run_rpa()

🚀  EXTRAÇÃO EM MASSA SAP — ZCO004

📂 dim_site carregado: 68 ordens

📦 Extraindo REAL...


com_error: (-2147352567, 'Exceção.', (619, 'SAP Frontend Server', 'The control could not be found by id.', 'C:\\Program Files (x86)\\SAP\\FrontEnd\\SapGui\\sapfront.HLP', 393215, 0), None)

In [ ]:
import win32com.client
import win32clipboard
import win32gui
import win32con
import pandas as pd
import subprocess
import time
from pathlib import Path

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

RAW_DIR  = BASE_PATH / "data" / "raw"
DIM_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

NOME_REAL        = "EXPORT_REAL.XLSX"
NOME_COMPROMISSO = "EXPORT_COMPROMIS.XLSX"

RAW_DIR.mkdir(parents=True, exist_ok=True)

# ======================================================
# HELPERS
# ======================================================

def set_clipboard_sap(lista_ordens):
    texto = "\r\n".join(lista_ordens)
    win32clipboard.OpenClipboard()
    try:
        win32clipboard.EmptyClipboard()
        win32clipboard.SetClipboardData(win32clipboard.CF_UNICODETEXT, texto)
    finally:
        win32clipboard.CloseClipboard()


def abrir_nova_sessao_sap(session_atual):
    """
    Abre nova sessão SAP (Ctrl+N) via sendVKey(74)
    Compatível com qualquer SAP GUI corporativo.
    """
    connection = session_atual.Parent
    qtd_antes = connection.Children.Count

    # Ctrl + N
    session_atual.findById("wnd[0]").sendVKey(74)
    time.sleep(2)

    while connection.Children.Count <= qtd_antes:
        time.sleep(0.5)

    nova_sessao = connection.Children(connection.Children.Count - 1)
    nova_sessao.findById("wnd[0]").maximize()
    nova_sessao.findById("wnd[0]").setFocus()

    print("   🆕 Nova sessão SAP aberta para COMPROMISSO.")
    return nova_sessao


def fechar_excel_e_focar_sap(session):
    subprocess.run(
        ["taskkill", "/f", "/im", "EXCEL.EXE"],
        capture_output=True,
        shell=True
    )
    time.sleep(2)

    def _focar(hwnd, _):
        if win32gui.IsWindowVisible(hwnd):
            titulo = win32gui.GetWindowText(hwnd)
            if "SAP" in titulo:
                win32gui.ShowWindow(hwnd, win32con.SW_MAXIMIZE)
                win32gui.SetForegroundWindow(hwnd)

    try:
        win32gui.EnumWindows(_focar, None)
    except:
        pass

    time.sleep(1)

    try:
        session.findById("wnd[0]").maximize()
        session.findById("wnd[0]").setFocus()
    except:
        pass

    time.sleep(1)
    print("   📊 Excel fechado. SAP maximizado e em foco.")


def exportar_sap_direto(session, nome_arquivo):
    try:
        grid_id = "wnd[0]/usr/cntlGRID1/shellcont/shell/shellcont[1]/shell"
        shell = session.findById(grid_id)
        shell.contextMenu()
        shell.selectContextMenuItem("&XXL")
        time.sleep(2)

        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(1)

        session.findById("wnd[1]/usr/ctxtDY_PATH").text      = str(RAW_DIR)
        session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = nome_arquivo
        session.findById("wnd[1]").sendVKey(0)

        time.sleep(4)
        print(f"   ✅ {nome_arquivo} exportado.")

    except Exception as e:
        print(f"   ⚠️ Erro na exportação ({nome_arquivo}): {e}")


def resumo_arquivo(caminho: Path, label: str):
    try:
        df = pd.read_excel(caminho)
        col_ordem = next((c for c in df.columns if "ordem" in str(c).lower()), None)
        col_valor = next((c for c in df.columns if "valor" in str(c).lower()), None)

        qtde  = df[col_ordem].nunique() if col_ordem else len(df)
        total = pd.to_numeric(df[col_valor], errors="coerce").sum() if col_valor else 0

        fmt = f"R$ {total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

        print(f"\n   📋 {label}")
        print(f"      Ordens únicas : {qtde:>6}")
        print(f"      Valor total   : {fmt}")

    except Exception as e:
        print(f"   ⚠️ Erro ao ler {label}: {e}")


def selecionar_layout_por_texto(session, texto_layout):
    """
    Seleciona o layout na janela de variantes buscando pelo texto,
    em vez de depender de posição fixa (lbl[x,y]).
    Mais robusto: não quebra se a tela renderizar diferente.
    """
    try:
        # Tenta encontrar o label pelo texto exato na janela de variantes
        for row in range(1, 20):
            for col in [1, 2, 3]:
                try:
                    lbl = session.findById(f"wnd[1]/usr/lbl[{col},{row}]")
                    if texto_layout.lower() in str(lbl.text).lower():
                        lbl.setFocus()
                        session.findById("wnd[1]").sendVKey(2)  # Enter duplo / selecionar
                        print(f"   ✅ Layout '{texto_layout}' encontrado em lbl[{col},{row}] e selecionado.")
                        return True
                except:
                    continue
        print(f"   ⚠️ Layout '{texto_layout}' não encontrado. Selecionando primeiro da lista.")
        session.findById("wnd[1]/usr/lbl[1,1]").setFocus()
        session.findById("wnd[1]").sendVKey(2)
        return False
    except Exception as e:
        print(f"   ⚠️ Erro ao selecionar layout: {e}")
        return False


# ======================================================
# MAIN
# ======================================================

def run_rpa():
    print("=" * 55)
    print("🚀  EXTRAÇÃO EM MASSA SAP — ZCO004")
    print("=" * 55)

    # ✅ CORREÇÃO: lê coluna "ordem" como string direto no read_excel,
    # e remove ".0" que aparece quando o Excel armazena números como float.
    df_site = pd.read_excel(DIM_SITE, dtype={"ordem": str})
    ordens = (
        df_site["ordem"]
        .dropna()
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)   # remove sufixo .0 caso exista
        .str.zfill(0)                            # mantém formato original sem padding
        .tolist()
    )
    print(f"\n📂 dim_site carregado: {len(ordens)} ordens")
    print(f"   Exemplo: {ordens[:3]}")           # debug visual para confirmar formato

    sap = win32com.client.GetObject("SAPGUI")
    session_real = sap.GetScriptingEngine.Children(0).Children(0)
    session_real.findById("wnd[0]").maximize()

    # ================= REAL =================
    session_real.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
    session_real.findById("wnd[0]").sendVKey(0)
    session_real.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

    session_real.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
    set_clipboard_sap(ordens)
    session_real.findById("wnd[1]/tbar[0]/btn[24]").press()
    session_real.findById("wnd[1]/tbar[0]/btn[8]").press()
    session_real.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(6)
    print("\n📦 Extraindo REAL...")

    # ✅ CORREÇÃO: usa busca por texto em vez de coordenada fixa lbl[62,6]
    session_real.findById("wnd[0]/tbar[1]/btn[7]").press()
    selecionar_layout_por_texto(session_real, "REAL")

    exportar_sap_direto(session_real, NOME_REAL)
    fechar_excel_e_focar_sap(session_real)
    session_real.findById("wnd[0]").sendVKey(12)

    # ================= COMPROMISSO =================
    print("\n📦 Extraindo COMPROMISSO em nova sessão...")
    session_comp = abrir_nova_sessao_sap(session_real)

    try:
        session_comp.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
        session_comp.findById("wnd[0]").sendVKey(0)
        session_comp.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

        session_comp.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
        set_clipboard_sap(ordens)
        session_comp.findById("wnd[1]/tbar[0]/btn[24]").press()
        session_comp.findById("wnd[1]/tbar[0]/btn[8]").press()
        session_comp.findById("wnd[0]/tbar[1]/btn[8]").press()

        time.sleep(10)

        # ✅ CORREÇÃO: mesma abordagem robusta para COMPROMISSO
        session_comp.findById("wnd[0]/tbar[1]/btn[7]").press()
        selecionar_layout_por_texto(session_comp, "COMPROMIS")

        exportar_sap_direto(session_comp, NOME_COMPROMISSO)
        fechar_excel_e_focar_sap(session_comp)

    finally:
        session_comp.findById("wnd[0]").close()
        print("   🧹 Sessão de COMPROMISSO encerrada.")

    # ================= RESUMO =================
    print("\n" + "=" * 55)
    print("📊  RESUMO DA EXTRAÇÃO")
    print("=" * 55)

    resumo_arquivo(RAW_DIR / NOME_REAL,        "REAL        (EXPORT_REAL.XLSX)")
    resumo_arquivo(RAW_DIR / NOME_COMPROMISSO, "COMPROMISSO (EXPORT_COMPROMIS.XLSX)")

    print("\n" + "=" * 55)
    print("✅  Processo finalizado!")
    print("=" * 55)


if __name__ == "__main__":
    run_rpa()

🚀  EXTRAÇÃO EM MASSA SAP — ZCO004

📂 dim_site carregado: 68 ordens
   Exemplo: ['606085', '605736', '605904']

📦 Extraindo REAL...
   ⚠️ Layout 'REAL' não encontrado. Selecionando primeiro da lista.
   ⚠️ Erro ao selecionar layout: (-2147352567, 'Exceção.', (619, 'SAP Frontend Server', 'The control could not be found by id.', 'C:\\Program Files (x86)\\SAP\\FrontEnd\\SapGui\\sapfront.HLP', 393215, 0), None)
   ⚠️ Erro na exportação (EXPORT_REAL.XLSX): (-2147352567, 'Exceção.', (619, 'SAP Frontend Server', 'The control could not be found by id.', 'C:\\Program Files (x86)\\SAP\\FrontEnd\\SapGui\\sapfront.HLP', 393215, 0), None)
   📊 Excel fechado. SAP maximizado e em foco.

📦 Extraindo COMPROMISSO em nova sessão...
   🆕 Nova sessão SAP aberta para COMPROMISSO.


In [6]:
import win32com.client
import win32clipboard
import win32gui
import win32con
import pandas as pd
import subprocess
import time
from pathlib import Path

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

RAW_DIR  = BASE_PATH / "data" / "raw"
DIM_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

NOME_REAL        = "EXPORT_REAL.XLSX"
NOME_COMPROMISSO = "EXPORT_COMPROMIS.XLSX"

RAW_DIR.mkdir(parents=True, exist_ok=True)

# ======================================================
# HELPERS
# ======================================================

def set_clipboard_sap(lista_ordens):
    texto = "\r\n".join(lista_ordens)
    win32clipboard.OpenClipboard()
    try:
        win32clipboard.EmptyClipboard()
        win32clipboard.SetClipboardData(win32clipboard.CF_UNICODETEXT, texto)
    finally:
        win32clipboard.CloseClipboard()


def abrir_nova_sessao_sap(session_atual):
    """
    Abre nova sessão SAP (Ctrl+N) via sendVKey(74)
    Compatível com qualquer SAP GUI corporativo.
    """
    connection = session_atual.Parent
    qtd_antes = connection.Children.Count

    # Ctrl + N
    session_atual.findById("wnd[0]").sendVKey(74)
    time.sleep(2)

    while connection.Children.Count <= qtd_antes:
        time.sleep(0.5)

    nova_sessao = connection.Children(connection.Children.Count - 1)
    nova_sessao.findById("wnd[0]").maximize()
    nova_sessao.findById("wnd[0]").setFocus()

    print("   🆕 Nova sessão SAP aberta para COMPROMISSO.")
    return nova_sessao


def fechar_excel_e_focar_sap(session):
    subprocess.run(
        ["taskkill", "/f", "/im", "EXCEL.EXE"],
        capture_output=True,
        shell=True
    )
    time.sleep(2)

    def _focar(hwnd, _):
        if win32gui.IsWindowVisible(hwnd):
            titulo = win32gui.GetWindowText(hwnd)
            if "SAP" in titulo:
                win32gui.ShowWindow(hwnd, win32con.SW_MAXIMIZE)
                win32gui.SetForegroundWindow(hwnd)

    try:
        win32gui.EnumWindows(_focar, None)
    except:
        pass

    time.sleep(1)

    try:
        session.findById("wnd[0]").maximize()
        session.findById("wnd[0]").setFocus()
    except:
        pass

    time.sleep(1)
    print("   📊 Excel fechado. SAP maximizado e em foco.")


def exportar_sap_direto(session, nome_arquivo):
    try:
        grid_id = "wnd[0]/usr/cntlGRID1/shellcont/shell/shellcont[1]/shell"
        shell = session.findById(grid_id)
        shell.contextMenu()
        shell.selectContextMenuItem("&XXL")
        time.sleep(2)

        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(1)

        session.findById("wnd[1]/usr/ctxtDY_PATH").text      = str(RAW_DIR)
        session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = nome_arquivo
        session.findById("wnd[1]").sendVKey(0)

        time.sleep(4)
        print(f"   ✅ {nome_arquivo} exportado.")

    except Exception as e:
        print(f"   ⚠️ Erro na exportação ({nome_arquivo}): {e}")


def resumo_arquivo(caminho: Path, label: str):
    try:
        df = pd.read_excel(caminho)
        col_ordem = next((c for c in df.columns if "ordem" in str(c).lower()), None)
        col_valor = next((c for c in df.columns if "valor" in str(c).lower()), None)

        qtde  = df[col_ordem].nunique() if col_ordem else len(df)
        total = pd.to_numeric(df[col_valor], errors="coerce").sum() if col_valor else 0

        fmt = f"R$ {total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

        print(f"\n   📋 {label}")
        print(f"      Ordens únicas : {qtde:>6}")
        print(f"      Valor total   : {fmt}")

    except Exception as e:
        print(f"   ⚠️ Erro ao ler {label}: {e}")

# ======================================================
# MAIN
# ======================================================

def run_rpa():
    print("=" * 55)
    print("🚀  EXTRAÇÃO EM MASSA SAP — ZCO004")
    print("=" * 55)

    df_site = pd.read_excel(DIM_SITE)
    ordens  = df_site["ordem"].dropna().astype("Int64").astype(str).str.strip().tolist()  # ← ajuste aqui
    print(f"\n📂 dim_site carregado: {len(ordens)} ordens")

    sap = win32com.client.GetObject("SAPGUI")
    session_real = sap.GetScriptingEngine.Children(0).Children(0)
    session_real.findById("wnd[0]").maximize()

    # ================= REAL =================
    session_real.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
    session_real.findById("wnd[0]").sendVKey(0)
    session_real.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

    session_real.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
    set_clipboard_sap(ordens)
    session_real.findById("wnd[1]/tbar[0]/btn[24]").press()
    session_real.findById("wnd[1]/tbar[0]/btn[8]").press()
    session_real.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(6)
    print("\n📦 Extraindo REAL...")

    session_real.findById("wnd[0]/usr/lbl[62,6]").setFocus()
    session_real.findById("wnd[0]/tbar[1]/btn[7]").press()
    session_real.findById("wnd[1]/usr/lbl[1,3]").setFocus()
    session_real.findById("wnd[1]").sendVKey(2)

    exportar_sap_direto(session_real, NOME_REAL)
    fechar_excel_e_focar_sap(session_real)
    session_real.findById("wnd[0]").sendVKey(12)

    # ================= COMPROMISSO =================
    print("\n📦 Extraindo COMPROMISSO em nova sessão...")
    session_comp = abrir_nova_sessao_sap(session_real)

    try:
        session_comp.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
        session_comp.findById("wnd[0]").sendVKey(0)
        session_comp.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

        session_comp.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
        set_clipboard_sap(ordens)
        session_comp.findById("wnd[1]/tbar[0]/btn[24]").press()
        session_comp.findById("wnd[1]/tbar[0]/btn[8]").press()
        session_comp.findById("wnd[0]/tbar[1]/btn[8]").press()

        time.sleep(10)

        session_comp.findById("wnd[0]/usr/lbl[78,6]").setFocus()
        session_comp.findById("wnd[0]/tbar[1]/btn[7]").press()
        session_comp.findById("wnd[1]/usr/lbl[1,1]").setFocus()
        session_comp.findById("wnd[1]").sendVKey(2)

        exportar_sap_direto(session_comp, NOME_COMPROMISSO)
        fechar_excel_e_focar_sap(session_comp)

    finally:
        session_comp.findById("wnd[0]").close()
        print("   🧹 Sessão de COMPROMISSO encerrada.")

    # ================= RESUMO =================
    print("\n" + "=" * 55)
    print("📊  RESUMO DA EXTRAÇÃO")
    print("=" * 55)

    resumo_arquivo(RAW_DIR / NOME_REAL,        "REAL        (EXPORT_REAL.XLSX)")
    resumo_arquivo(RAW_DIR / NOME_COMPROMISSO, "COMPROMISSO (EXPORT_COMPROMIS.XLSX)")

    print("\n" + "=" * 55)
    print("✅  Processo finalizado!")
    print("=" * 55)


if __name__ == "__main__":
    run_rpa()

🚀  EXTRAÇÃO EM MASSA SAP — ZCO004

📂 dim_site carregado: 75 ordens

📦 Extraindo REAL...
   ✅ EXPORT_REAL.XLSX exportado.
   📊 Excel fechado. SAP maximizado e em foco.

📦 Extraindo COMPROMISSO em nova sessão...
   🆕 Nova sessão SAP aberta para COMPROMISSO.
   ✅ EXPORT_COMPROMIS.XLSX exportado.
   📊 Excel fechado. SAP maximizado e em foco.
   🧹 Sessão de COMPROMISSO encerrada.

📊  RESUMO DA EXTRAÇÃO

   📋 REAL        (EXPORT_REAL.XLSX)
      Ordens únicas :     75
      Valor total   : R$ 45.387.847,02

   📋 COMPROMISSO (EXPORT_COMPROMIS.XLSX)
      Ordens únicas :     57
      Valor total   : R$ 3.600.134,12

✅  Processo finalizado!


In [7]:
import win32com.client
import win32clipboard
import win32gui
import win32con
import pandas as pd
import subprocess
import shutil
import time
from pathlib import Path
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

RAW_DIR      = BASE_PATH / "data" / "raw"
HISTORICO_DIR = BASE_PATH / "data" / "historico"
DIM_SITE     = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

# Nomes padrão que o pipeline lê (não mudam)
NOME_REAL        = "EXPORT_REAL.XLSX"
NOME_COMPROMISSO = "EXPORT_COMPROMIS.XLSX"

RAW_DIR.mkdir(parents=True, exist_ok=True)
HISTORICO_DIR.mkdir(parents=True, exist_ok=True)

# ======================================================
# HELPERS
# ======================================================

def get_data_hoje():
    return datetime.now().strftime("%Y-%m-%d")


def salvar_com_historico(nome_padrao: str):
    """
    Copia o arquivo recém-exportado para a pasta histórico
    com sufixo de data. O arquivo padrão em /raw permanece
    intacto para o pipeline consumir normalmente.

    Ex: EXPORT_REAL.XLSX → historico/EXPORT_REAL_2025-01-31.XLSX
    """
    origem  = RAW_DIR / nome_padrao
    stem    = Path(nome_padrao).stem          # "EXPORT_REAL"
    sufixo  = Path(nome_padrao).suffix        # ".XLSX"
    destino = HISTORICO_DIR / f"{stem}_{get_data_hoje()}{sufixo}"

    if origem.exists():
        shutil.copy2(origem, destino)
        print(f"   📁 Histórico salvo: historico/{destino.name}")
    else:
        print(f"   ⚠️  Arquivo não encontrado para histórico: {origem}")


def set_clipboard_sap(lista_ordens):
    texto = "\r\n".join(lista_ordens)
    win32clipboard.OpenClipboard()
    try:
        win32clipboard.EmptyClipboard()
        win32clipboard.SetClipboardData(win32clipboard.CF_UNICODETEXT, texto)
    finally:
        win32clipboard.CloseClipboard()


def abrir_nova_sessao_sap(session_atual):
    connection = session_atual.Parent
    qtd_antes  = connection.Children.Count

    session_atual.findById("wnd[0]").sendVKey(74)
    time.sleep(2)

    while connection.Children.Count <= qtd_antes:
        time.sleep(0.5)

    nova_sessao = connection.Children(connection.Children.Count - 1)
    nova_sessao.findById("wnd[0]").maximize()
    nova_sessao.findById("wnd[0]").setFocus()

    print("   🆕 Nova sessão SAP aberta para COMPROMISSO.")
    return nova_sessao


def fechar_excel_e_focar_sap(session):
    subprocess.run(
        ["taskkill", "/f", "/im", "EXCEL.EXE"],
        capture_output=True,
        shell=True
    )
    time.sleep(2)

    def _focar(hwnd, _):
        if win32gui.IsWindowVisible(hwnd):
            titulo = win32gui.GetWindowText(hwnd)
            if "SAP" in titulo:
                win32gui.ShowWindow(hwnd, win32con.SW_MAXIMIZE)
                win32gui.SetForegroundWindow(hwnd)

    try:
        win32gui.EnumWindows(_focar, None)
    except:
        pass

    time.sleep(1)

    try:
        session.findById("wnd[0]").maximize()
        session.findById("wnd[0]").setFocus()
    except:
        pass

    time.sleep(1)
    print("   📊 Excel fechado. SAP maximizado e em foco.")


def exportar_sap_direto(session, nome_arquivo):
    try:
        grid_id = "wnd[0]/usr/cntlGRID1/shellcont/shell/shellcont[1]/shell"
        shell   = session.findById(grid_id)
        shell.contextMenu()
        shell.selectContextMenuItem("&XXL")
        time.sleep(2)

        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(1)

        session.findById("wnd[1]/usr/ctxtDY_PATH").text     = str(RAW_DIR)
        session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = nome_arquivo
        session.findById("wnd[1]").sendVKey(0)

        time.sleep(4)
        print(f"   ✅ {nome_arquivo} exportado.")

    except Exception as e:
        print(f"   ⚠️ Erro na exportação ({nome_arquivo}): {e}")


def resumo_arquivo(caminho: Path, label: str):
    try:
        df = pd.read_excel(caminho)
        col_ordem = next((c for c in df.columns if "ordem" in str(c).lower()), None)
        col_valor = next((c for c in df.columns if "valor" in str(c).lower()), None)

        qtde  = df[col_ordem].nunique() if col_ordem else len(df)
        total = pd.to_numeric(df[col_valor], errors="coerce").sum() if col_valor else 0

        fmt = f"R$ {total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

        print(f"\n   📋 {label}")
        print(f"      Ordens únicas : {qtde:>6}")
        print(f"      Valor total   : {fmt}")

    except Exception as e:
        print(f"   ⚠️ Erro ao ler {label}: {e}")

# ======================================================
# MAIN
# ======================================================

def run_rpa():
    print("=" * 55)
    print("🚀  EXTRAÇÃO EM MASSA SAP — ZCO004")
    print(f"📅  Data: {get_data_hoje()}")
    print("=" * 55)

    df_site = pd.read_excel(DIM_SITE)
    ordens  = df_site["ordem"].dropna().astype("Int64").astype(str).str.strip().tolist()
    print(f"\n📂 dim_site carregado: {len(ordens)} ordens")

    sap          = win32com.client.GetObject("SAPGUI")
    session_real = sap.GetScriptingEngine.Children(0).Children(0)
    session_real.findById("wnd[0]").maximize()

    # ================= REAL =================
    session_real.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
    session_real.findById("wnd[0]").sendVKey(0)
    session_real.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

    session_real.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
    set_clipboard_sap(ordens)
    session_real.findById("wnd[1]/tbar[0]/btn[24]").press()
    session_real.findById("wnd[1]/tbar[0]/btn[8]").press()
    session_real.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(6)
    print("\n📦 Extraindo REAL...")

    session_real.findById("wnd[0]/usr/lbl[62,6]").setFocus()
    session_real.findById("wnd[0]/tbar[1]/btn[7]").press()
    session_real.findById("wnd[1]/usr/lbl[1,3]").setFocus()
    session_real.findById("wnd[1]").sendVKey(2)

    exportar_sap_direto(session_real, NOME_REAL)
    salvar_com_historico(NOME_REAL)           # ← cópia datada em /historico
    fechar_excel_e_focar_sap(session_real)
    session_real.findById("wnd[0]").sendVKey(12)

    # ================= COMPROMISSO =================
    print("\n📦 Extraindo COMPROMISSO em nova sessão...")
    session_comp = abrir_nova_sessao_sap(session_real)

    try:
        session_comp.findById("wnd[0]/tbar[0]/okcd").text = "zco004"
        session_comp.findById("wnd[0]").sendVKey(0)
        session_comp.findById("wnd[0]/usr/txt$6-KOKRS").text = "1000"

        session_comp.findById("wnd[0]/usr/btn%__6ORDGRP_%_APP_%-VALU_PUSH").press()
        set_clipboard_sap(ordens)
        session_comp.findById("wnd[1]/tbar[0]/btn[24]").press()
        session_comp.findById("wnd[1]/tbar[0]/btn[8]").press()
        session_comp.findById("wnd[0]/tbar[1]/btn[8]").press()

        time.sleep(10)

        session_comp.findById("wnd[0]/usr/lbl[78,6]").setFocus()
        session_comp.findById("wnd[0]/tbar[1]/btn[7]").press()
        session_comp.findById("wnd[1]/usr/lbl[1,1]").setFocus()
        session_comp.findById("wnd[1]").sendVKey(2)

        exportar_sap_direto(session_comp, NOME_COMPROMISSO)
        salvar_com_historico(NOME_COMPROMISSO)    # ← cópia datada em /historico
        fechar_excel_e_focar_sap(session_comp)

    finally:
        session_comp.findById("wnd[0]").close()
        print("   🧹 Sessão de COMPROMISSO encerrada.")

    # ================= RESUMO =================
    print("\n" + "=" * 55)
    print("📊  RESUMO DA EXTRAÇÃO")
    print("=" * 55)

    resumo_arquivo(RAW_DIR / NOME_REAL,        "REAL        (EXPORT_REAL.XLSX)")
    resumo_arquivo(RAW_DIR / NOME_COMPROMISSO, "COMPROMISSO (EXPORT_COMPROMIS.XLSX)")

    print("\n" + "=" * 55)
    print(f"📁  Histórico salvo em: data/historico/")
    print("✅  Processo finalizado!")
    print("=" * 55)


if __name__ == "__main__":
    run_rpa()

🚀  EXTRAÇÃO EM MASSA SAP — ZCO004
📅  Data: 2026-05-15

📂 dim_site carregado: 75 ordens

📦 Extraindo REAL...
   ✅ EXPORT_REAL.XLSX exportado.
   📁 Histórico salvo: historico/EXPORT_REAL_2026-05-15.XLSX
   📊 Excel fechado. SAP maximizado e em foco.

📦 Extraindo COMPROMISSO em nova sessão...
   🆕 Nova sessão SAP aberta para COMPROMISSO.
   ✅ EXPORT_COMPROMIS.XLSX exportado.
   📁 Histórico salvo: historico/EXPORT_COMPROMIS_2026-05-15.XLSX
   📊 Excel fechado. SAP maximizado e em foco.
   🧹 Sessão de COMPROMISSO encerrada.

📊  RESUMO DA EXTRAÇÃO

   📋 REAL        (EXPORT_REAL.XLSX)
      Ordens únicas :     75
      Valor total   : R$ 45.387.847,02

   📋 COMPROMISSO (EXPORT_COMPROMIS.XLSX)
      Ordens únicas :     57
      Valor total   : R$ 3.620.848,56

📁  Histórico salvo em: data/historico/
✅  Processo finalizado!
